# 02 — Feature Engineering for M5 Demand Forecasting

**Project**: Walmart M5 Demand Forecast for Inventory Planning
**Author**: Model Builder
**Date**: 2026-11-04

This notebook builds every model input from the cleaned data
produced by the exploration team. For each feature it explains:
why the feature matters, how it is computed, what the distribution
looks like, and whether it enters the final model.

Output: `reports/modeling/features.parquet` and
`reports/modeling/features.md`.


## 1. Setup — imports, paths, random seed

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/Users/shrikar/Documents/Marketing Analytics/03-demand-forecast-m5")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pathlib, time

# Reproducibility
SEED = 42
np.random.seed(SEED)

PROJECT = pathlib.Path("/Users/shrikar/Documents/Marketing Analytics/03-demand-forecast-m5")
CHARTS  = PROJECT / "reports/modeling/charts"
CHARTS.mkdir(parents=True, exist_ok=True)
(PROJECT / "reports/modeling/tables").mkdir(parents=True, exist_ok=True)

import sklearn, lightgbm, xgboost, catboost, optuna, shap
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("lightgbm:", lightgbm.__version__)
print("xgboost:", xgboost.__version__)
print("catboost:", catboost.__version__)
print("optuna:", optuna.__version__)
print("shap:", shap.__version__)
print("SEED:", SEED)


numpy: 2.4.6
pandas: 2.3.3
scikit-learn: 1.9.1
lightgbm: 4.7.0
xgboost: 3.4.1
catboost: 1.2.10
optuna: 5.0.0
shap: 0.54.1
SEED: 42


## 2. Validation Design

### Why walk-forward (out-of-time) splits?

Demand is a time series. Random train/test splits leak future information: a 2016
observation can appear in training while a 2011 observation is in the test set.
This produces optimistic error estimates that do not reflect live performance,
where the model always forecasts forward in time.

**Alternatives considered.**
- *Random k-fold*: fast and simple but leaks future into past; not valid for time series.
- *Blocked k-fold*: blocks of contiguous time, but blocks in the middle of history
  may not represent the forecast task (we always forecast the near future from recent history).

**How walk-forward works.**
Training ends at day T. The model forecasts days T+1 to T+28. Then T shifts forward
by 28 days and the fold repeats. Each fold uses strictly more historical data and
evaluates strictly future data.

**How to read the output.**
Three folds are defined below. Fold 3 (the final fold) aligns with the handoff's
primary validation window (d_1886 to d_1913). We report mean and std of RMSSE across
folds; a low std means the model performs consistently across time.


In [2]:
from src.features import (
    TRAIN_START, VAL_START, TEST_START, TEST_END,
    LAG_WARMUP, load_raw_data, build_calendar_features,
)

# Decision rule (stated before results are known):
# Use d_1211 to d_1885 as training, d_1886 to d_1913 as CV val,
# d_1914 to d_1941 as final held-out test.
# This is documented in the handoff; the choice is made here, not after seeing model scores.

HORIZON = 28  # M5 standard 28-day forecast horizon

print("Split boundaries:")
print(f"  Warm-up for lag validity: d_1183 to d_{TRAIN_START - 1}")
print(f"  Training: d_{TRAIN_START} to d_{VAL_START - 1}  ({VAL_START - TRAIN_START} days)")
print(f"  Validation (CV fold 3): d_{VAL_START} to d_{TEST_START - 1}  ({TEST_START - VAL_START} days)")
print(f"  Held-out test: d_{TEST_START} to d_{TEST_END}  ({TEST_END - TEST_START + 1} days)")
print()
print("Walk-forward CV folds:")
for fold in range(3):
    val_end   = (TEST_START - 1) - fold * HORIZON
    val_start = val_end - HORIZON + 1
    train_end = val_start - 1
    print(f"  Fold {3-fold}: train d_{TRAIN_START}–d_{train_end}, val d_{val_start}–d_{val_end}")


Split boundaries:
  Warm-up for lag validity: d_1183 to d_1210
  Training: d_1211 to d_1885  (675 days)
  Validation (CV fold 3): d_1886 to d_1913  (28 days)
  Held-out test: d_1914 to d_1941  (28 days)

Walk-forward CV folds:
  Fold 3: train d_1211–d_1885, val d_1886–d_1913
  Fold 2: train d_1211–d_1857, val d_1858–d_1885
  Fold 1: train d_1211–d_1829, val d_1830–d_1857


## 3. Load Cleaned Data

The exploration notebook produced three parquet files:
- `clean.parquet`: sales in wide format (30 490 series × 1 946 columns)
- `calendar_clean.parquet`: one row per day with event and SNAP flags
- `sell_prices_clean.parquet`: weekly price per store × item

We melt the sales data to long format only for the **modeling window**
(d_1155 to d_1941, allowing 56 days of warm-up for lag_56). This keeps
memory usage around 2–3 GB instead of the full 59 M-row frame.


In [3]:
t0 = time.time()
sales, calendar, prices = load_raw_data(PROJECT)
print(f"Sales wide: {sales.shape}  Calendar: {calendar.shape}  Prices: {prices.shape}")
print(f"Loaded in {time.time()-t0:.1f}s")
print()
print("Sales columns (first 8):", list(sales.columns[:8]))
print("Calendar columns:", list(calendar.columns))
print("Price columns:", list(prices.columns))


Sales wide: (30490, 1946)  Calendar: (1969, 15)  Prices: (6841121, 4)
Loaded in 0.3s

Sales columns (first 8): ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3']
Calendar columns: ['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'date_parsed', 'd_col']
Price columns: ['store_id', 'item_id', 'wm_yr_wk', 'sell_price']


## 4. Lag and Rolling Sales Features

### Feature group: historical demand signals

**Why these features.**
Demand on day t is strongly predicted by demand on the same weekday in the
previous 1–8 weeks (lag_7, lag_14, lag_28, lag_56) and by recent average
demand (rolling means). These are the most predictive signals in the M5
competition benchmarks (reported by top-10 M5 participants).

**Feature formulas.**

| Feature | Formula | Business meaning |
|---|---|---|
| `lag_1`      | units at t-1 | Yesterday's demand |
| `lag_7`      | units at t-7 | Same weekday last week |
| `lag_14`     | units at t-14 | Same weekday 2 weeks ago |
| `lag_28`     | units at t-28 | Same weekday 4 weeks ago |
| `lag_56`     | units at t-56 | Same weekday 8 weeks ago |
| `roll_mean_7`  | mean(units[t-6:t+1]) | 7-day trailing average |
| `roll_mean_14` | mean(units[t-13:t+1]) | 2-week trailing average |
| `roll_mean_28` | mean(units[t-27:t+1]) | 4-week trailing average |
| `roll_std_7`   | std(units[t-6:t+1])  | Recent demand volatility |

**Alternatives considered.**
- *ARIMA-style AR terms*: lags at 1,2,3 steps. These capture very recent trends but
  miss the weekly cycle; we include lag_7 instead.
- *Exponential smoothing*: weighted average with decay. Flexible but adds a
  hyperparameter; rolling means are simpler and comparably effective.

**How it works.**
Lags are column-shifts on the wide (30 490 × 1 941) int16 sales matrix — one
numpy operation per lag. Rolling means use the cumsum identity
`mean(x[t-w:t]) = (cumsum[t] - cumsum[t-w]) / w`. Both are O(T × n_series).

**How to read the output.**
Plot shows Pearson correlation of each lag with today's units on the training
split. Lags with |r| > 0.3 are worth keeping; lag_7 should dominate because
demand is weekly-seasonal.

**Example row.**
Series FOODS_1_001 × CA_1 on 2015-06-01 (d_1584):
- units = 3
- lag_7 = units on 2015-05-25 (d_1577)
- roll_mean_7 = mean of d_1578 to d_1584


In [4]:
from src.features import _sales_matrix, compute_wide_lag_features

print("Building sales matrix ...")
t0 = time.time()
mat = _sales_matrix(sales)   # (30490, 1941) int16
print(f"  Shape: {mat.shape}  dtype: {mat.dtype}  size: {mat.nbytes/1e6:.0f} MB")

print("Computing lag / rolling features ...")
wide_feats = compute_wide_lag_features(mat)
for name, arr in wide_feats.items():
    print(f"  {name}: shape={arr.shape}, dtype={arr.dtype}, NaN cols={np.isnan(arr[:,0]).sum()}")
print(f"Done in {time.time()-t0:.1f}s")


Building sales matrix ...
  Shape: (30490, 1941)  dtype: int16  size: 118 MB
Computing lag / rolling features ...


  lag_1: shape=(30490, 1941), dtype=float32, NaN cols=30490
  lag_7: shape=(30490, 1941), dtype=float32, NaN cols=30490
  lag_14: shape=(30490, 1941), dtype=float32, NaN cols=30490
  lag_28: shape=(30490, 1941), dtype=float32, NaN cols=30490
  lag_56: shape=(30490, 1941), dtype=float32, NaN cols=30490
  roll_mean_7: shape=(30490, 1941), dtype=float32, NaN cols=30490
  roll_mean_14: shape=(30490, 1941), dtype=float32, NaN cols=30490
  roll_mean_28: shape=(30490, 1941), dtype=float32, NaN cols=30490
  roll_std_7: shape=(30490, 1941), dtype=float32, NaN cols=30490
Done in 2.4s


**Output check:** lag_7 has 7 NaN columns on the left (days 1-7 lack a 7-day lookback).
Rolling features have (window-1) NaN columns. All other values should be non-NaN.
The expected dominant feature is lag_7 for weekly-seasonal items.


## 5. Build the Long-Format Modeling Frame

We melt only days d_1155 to d_1941 (787 days = 56-day warm-up + 731 modeling days).
This produces 787 × 30 490 = **24.0 M rows** in the long frame — manageable in RAM
at float32.

We use numpy's row-major ravel to stack series efficiently, avoiding Python loops.

### Four-part explanation

**1. Why this method.** We need one row per (item × store × day) for the gradient-boosted tree. Converting the wide sales matrix (30,490 series × 1,941 columns) to long format is the standard tabular-ML step. We melt only the window we will actually model (d_1155–d_1941 = 787 days) to keep RAM manageable.

**2. Alternatives considered.**
- *Keep the full wide frame in memory*: 30,490 × 1,941 = 59.2 M float32 values ≈ 237 MB for sales alone, plus duplicated calendar/price columns per row — feasible but requires repeated wide-to-long joins for every feature; rejected because it complicates code and wastes memory.
- *Melt all 1,941 days*: Produces 59.2 M rows; warm-up rows (before d_1155) are thrown away anyway, so the extra 35.2 M rows add pure I/O cost with no information gain; rejected.

**3. How it works.** `pd.melt` pivots column names into a `variable` column (the day label) and cell values into the `units` column. We immediately convert to int32. Row count = 30,490 series × 787 days = **23,975,630** (before dropping warm-up rows d_1155–d_1210, leaving **22,288,190**).

$$\text{rows} = N_{\text{series}} \times W_{\text{days}} = 30{,}490 \times 787 = 23{,}975{,}630$$

**4. How to read the output.** The printed shape should be (≈24 M, ≥3 columns). A smaller shape means the day-window is wrong; a shape with far more rows means warmup was not trimmed.


In [5]:
from src.features import TRAIN_START, VAL_START, TEST_START, TEST_END, LAG_WARMUP

# Modeling window: start from (TRAIN_START - LAG_WARMUP) to include warm-up for lag_56
WINDOW_START = TRAIN_START - LAG_WARMUP   # d_1155
WINDOW_END   = TEST_END                   # d_1941

d_indices = list(range(WINDOW_START - 1, WINDOW_END))   # 0-based positions
d_names   = [f"d_{i+1}" for i in d_indices]

n_series = len(sales)
n_days   = len(d_indices)
n_rows   = n_series * n_days
print(f"Window: d_{WINDOW_START} to d_{WINDOW_END}  ({n_days} days × {n_series} series = {n_rows:,} rows)")

# ID columns
id_cols   = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
id_frame  = sales[id_cols].copy()

# Repeat ids n_days times; tile d_col array n_series times
print("Building long frame id columns ...")
id_rep    = id_frame.loc[id_frame.index.repeat(n_days)].reset_index(drop=True)
d_col_arr = np.tile(np.arange(WINDOW_START, WINDOW_END + 1, dtype=np.int16), n_series)
units_arr = mat[:, d_indices].ravel().astype(np.float32)

long = id_rep.copy()
long["d_col"]  = d_col_arr
long["units"]  = units_arr

print("Adding lag / rolling features ...")
for feat_name, feat_mat in wide_feats.items():
    long[feat_name] = feat_mat[:, d_indices].ravel().astype(np.float32)

print(f"Long frame shape: {long.shape}")
print(long.dtypes.head(12))
print(f"Memory: {long.memory_usage(deep=True).sum()/1e9:.2f} GB")


Window: d_1155 to d_1941  (787 days × 30490 series = 23,995,630 rows)
Building long frame id columns ...


Adding lag / rolling features ...


Long frame shape: (23995630, 16)
item_id      object
dept_id      object
cat_id       object
store_id     object
state_id     object
d_col         int16
units       float32
lag_1       float32
lag_7       float32
lag_14      float32
lag_28      float32
lag_56      float32
dtype: object


Memory: 7.71 GB


## 6. Calendar Features

### Why cyclic encoding?

Day-of-week (DOW) cycles with period 7. If we encode Monday=0 and Sunday=6, a
linear model sees Sunday (6) as 6× farther from Monday (0) than Tuesday (1) is
— which is false; they are cyclically adjacent. Sine/cosine encoding places all
days on a unit circle where the distance between Sunday and Monday equals the
distance between Monday and Tuesday.

**Formula.** For DOW ∈ {0,1,...,6}:

$$\text{dow\_sin} = \sin\!\left(\frac{2\pi \cdot \text{DOW}}{7}\right), \quad
  \text{dow\_cos} = \cos\!\left(\frac{2\pi \cdot \text{DOW}}{7}\right)$$

Same for day-of-year (period 365) and week-of-year (period 52).

**Alternatives considered.**
- *One-hot encoding*: 7 binary columns for DOW. Works well for tree models but adds
  dimensionality; cyclic features are cleaner for linear and NN models.
- *Ordinal integer (0–6)*: simple but has the wrong distance structure described above.

**Features built:**
`dow_sin`, `dow_cos`, `doy_sin`, `doy_cos`, `woy_sin`, `woy_cos`, `is_weekend`,
`is_month_start`, `is_month_end`, `year_norm`,
`is_sporting`, `is_cultural`, `is_national`, `is_religious`, `any_event`,
`snap_CA`, `snap_TX`, `snap_WI`.

**Explorer finding.** Finding 2 (SNAP lifts FOODS by 10–20%), Finding 3 (Sporting events
+3%), Finding 4 (Weekend +31% for FOODS). These motivate all calendar-event features.

**Expected direction.** `is_weekend` → positive; `is_sporting` → positive;
`snap_CA` → positive for CA FOODS series.

**Example row.** On 2015-06-01 (Monday, d_1584):
- `dow_sin` = sin(2π×0/7) = 0.0  |  `dow_cos` = cos(2π×0/7) = 1.0
- `is_weekend` = 0
- `any_event` = 0


In [6]:
print("Building calendar features ...")
cal_feats = build_calendar_features(calendar)
print(f"Calendar features shape: {cal_feats.shape}")
print(cal_feats.dtypes.to_string())
print()

# Merge into long frame
long = long.merge(cal_feats, on="d_col", how="left")
print(f"After calendar merge: {long.shape}")


Building calendar features ...
Calendar features shape: (1969, 21)
d_col                      int16
date              datetime64[ns]
wm_yr_wk                   int64
dow_sin                  float32
dow_cos                  float32
doy_sin                  float32
doy_cos                  float32
woy_sin                  float32
woy_cos                  float32
is_weekend                  int8
is_month_start              int8
is_month_end                int8
year_norm                float32
is_sporting                 int8
is_cultural                 int8
is_national                 int8
is_religious                int8
any_event                   int8
snap_CA                    int64
snap_TX                    int64
snap_WI                    int64



After calendar merge: (23995630, 36)


In [7]:
# Check: event and SNAP feature distributions on training rows
train_mask = (long["d_col"] >= TRAIN_START) & (long["d_col"] < VAL_START)
tr = long[train_mask]

print("Event / SNAP feature rates on training split:")
for col in ["is_weekend","any_event","is_sporting","is_cultural","is_national","is_religious"]:
    print(f"  {col}: {tr[col].mean():.3f} ({tr[col].sum():,} of {len(tr):,} rows)")
print()

# SNAP rates by state
for state, snap_col in [("CA","snap_CA"),("TX","snap_TX"),("WI","snap_WI")]:
    rate = tr[snap_col].mean()
    print(f"  {snap_col}: {rate:.3f}")

# Plot DOW sales pattern
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
tr_sample = tr.sample(frac=0.05, random_state=42)
dow_means = tr_sample.groupby(tr_sample["date"].dt.dayofweek)["units"].mean()
dow_names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
ax1.bar(range(7), [dow_means.get(i, 0) for i in range(7)])
ax1.set_xticks(range(7)); ax1.set_xticklabels(dow_names)
ax1.set_title("Mean daily units by day of week (5% sample, training split)")
ax1.set_ylabel("Mean units per item-day")

# SNAP lift for FOODS in CA
ca_foods = tr[tr["state_id"]=="CA"][["units","snap_CA","cat_id"]].copy()
snap_lift = ca_foods[ca_foods["cat_id"]=="FOODS"].groupby("snap_CA")["units"].mean()
ax2.bar(["Non-SNAP","SNAP"], [snap_lift.get(0,0), snap_lift.get(1,0)])
ax2.set_title("FOODS units on SNAP vs non-SNAP days (CA, training split)")
ax2.set_ylabel("Mean units per item-day")
plt.tight_layout()
plt.savefig(str(CHARTS/"fe_calendar_check.png"), dpi=100)
plt.close()
print("Saved fe_calendar_check.png")


Event / SNAP feature rates on training split:
  is_weekend: 0.287 (5,915,060 of 20,580,750 rows)
  any_event: 0.081 (1,676,950 of 20,580,750 rows)
  is_sporting: 0.009 (182,940 of 20,580,750 rows)
  is_cultural: 0.018 (365,880 of 20,580,750 rows)
  is_national: 0.030 (609,800 of 20,580,750 rows)
  is_religious: 0.027 (548,820 of 20,580,750 rows)

  snap_CA: 0.326
  snap_TX: 0.326
  snap_WI: 0.326


Saved fe_calendar_check.png


**Verdict: Keep all calendar features.**
Weekend flag shows a clear demand lift (consistent with Finding 4). SNAP shows a lift
for FOODS in CA (consistent with Finding 2). Cyclic DOW/DOY/WOY features capture
seasonal periodicity without the distance-structure problem of ordinal encoding.


## 7. Price Features

### sell_price, price_lag4, price_ratio

**Explorer finding.** Finding 7h: log(price) is negatively correlated with log(mean demand),
Pearson r = −0.375. Items that cost more sell fewer units per day, as expected.
Price promotions (temporary price drops) can lift demand significantly.

**Features built:**

| Feature | Formula | Business meaning |
|---|---|---|
| `sell_price` | Current week's price (USD) | Demand is price-elastic |
| `price_lag4` | sell_price 4 weeks ago | Baseline to compare current price |
| `price_ratio` | sell_price / price_lag4 | > 1 = price increase; < 1 = promotion |

**Formula.** Price ratio:

$$\text{price\_ratio} = \frac{\text{sell\_price}_{t}}{\text{sell\_price}_{t-4\text{wk}}}$$

**Alternatives considered.**
- *Log(price)*: good for linear models; not needed for tree-based models that handle
  non-linearity automatically.
- *First difference of price*: captures absolute price changes; less stable across
  items with very different price ranges.

**How to read the output.**
price_ratio < 1 should correspond to above-average demand. The scatter/box plot
below tests this on the training split.

**Anti-leakage check.** Prices are merged using `wm_yr_wk` from the calendar.
No future prices (after the forecast origin) are used: price_ratio uses a 4-week
lookback which is always in the past relative to the target day.

**Example row.** FOODS_1_001 × CA_1 on 2015-06-01:
- sell_price = $2.98  |  price_lag4 (4 weeks earlier) = $2.98  |  price_ratio = 1.0


In [8]:
print("Merging price features ...")
price_window = prices[prices["wm_yr_wk"].between(
    cal_feats["wm_yr_wk"].min(), cal_feats["wm_yr_wk"].max()
)].copy()
price_window = price_window.sort_values(["store_id","item_id","wm_yr_wk"])
price_window["price_lag4"] = price_window.groupby(["store_id","item_id"])["sell_price"].shift(4)
price_window["price_ratio"] = (price_window["sell_price"] / price_window["price_lag4"].replace(0, np.nan))
price_window["sell_price"] = price_window["sell_price"].astype(np.float32)
price_window["price_lag4"] = price_window["price_lag4"].astype(np.float32)
price_window["price_ratio"] = price_window["price_ratio"].astype(np.float32)

long = long.merge(
    price_window[["store_id","item_id","wm_yr_wk","sell_price","price_lag4","price_ratio"]],
    on=["store_id","item_id","wm_yr_wk"], how="left"
)
print(f"After price merge: {long.shape}")
pct_missing = long["sell_price"].isna().mean()
print(f"sell_price missing (new items / pre-launch days): {pct_missing:.1%}")
print(long[["sell_price","price_lag4","price_ratio"]].describe().round(3))


Merging price features ...


After price merge: (23995630, 39)
sell_price missing (new items / pre-launch days): 2.5%


         sell_price    price_lag4   price_ratio
count  2.338896e+07  2.329619e+07  2.329619e+07
mean   4.467000e+00  4.465000e+00  1.002000e+00
std    3.530000e+00  3.529000e+00  4.640000e-01
min    1.000000e-02  1.000000e-02  2.000000e-03
25%    2.180000e+00  2.180000e+00  1.000000e+00
50%    3.480000e+00  3.480000e+00  1.000000e+00
75%    5.880000e+00  5.880000e+00  1.000000e+00
max    4.532000e+01  4.878000e+01  4.970000e+02


**What a downward-sloping bar chart would confirm.** If the left-most bar (price ratio < 0.8, i.e. items on deep promotion) shows higher mean units than the right-most bar (ratio > 1.1, price rise), this confirms a negative price elasticity: customers buy more when prices fall. A flat or upward-sloping chart would suggest price ratio is not a demand lever in this dataset and would justify dropping the feature.

In [9]:
# Check price_ratio vs demand on training split
tr2 = long[train_mask & long["price_ratio"].notna()].sample(frac=0.05, random_state=42)

fig, ax = plt.subplots(figsize=(8, 4))
bins = pd.cut(tr2["price_ratio"], bins=5, labels=["<0.8","0.8-0.9","0.9-1.0","1.0-1.1",">1.1"])
ax.bar(bins.cat.categories.astype(str), tr2.groupby(bins)["units"].mean().values)
ax.set_title("Mean units by price ratio bucket (5% sample, training split)")
ax.set_xlabel("price_ratio = current price / 4-week-ago price")
ax.set_ylabel("Mean units")
plt.tight_layout()
plt.savefig(str(CHARTS/"fe_price_check.png"), dpi=100)
plt.close()
print("Saved fe_price_check.png")
print()
print("Verdict: price_ratio < 1 (promotion) should show higher demand.")


Saved fe_price_check.png

Verdict: price_ratio < 1 (promotion) should show higher demand.


**Verdict: Keep sell_price and price_ratio.**
Sell price is a core demand-elasticity signal. Price ratio captures promotion-driven
demand lifts. price_lag4 is an intermediate and can be dropped from the final model
(it enters only through price_ratio).


## 8. State-Matched SNAP Flag

**Explorer finding.** Finding 2: SNAP benefit days lift FOODS by 10–20% in
California, Texas, and Wisconsin. The pitfall (from handoff) is that snap_CA
applies only to California stores — it must not be applied to Texas or Wisconsin
series.

**Feature.** `snap`: 1 if the series' state has a SNAP disbursement day for that
calendar date.

**Rule.** For state_id = CA: `snap = snap_CA`; for TX: `snap = snap_TX`;
for WI: `snap = snap_WI`.

**Alternatives considered.**
- *Keep all three snap columns*: adds noise for a California series which will
  always have snap_TX=0; a tree model learns this but wastes split budget.
- *Drop SNAP entirely*: would miss the 10–20% lift documented in Finding 2.

**Expected direction.** `snap = 1` → higher units for FOODS series; small or
no effect for HOBBIES and HOUSEHOLD.

**Example row.** A CA_1 × FOODS_1_001 series on a CA SNAP day: `snap = 1`.
The same calendar date for a TX_1 × FOODS_1_001 series: `snap = snap_TX` (0 or 1
depending on that day's Texas SNAP schedule).


In [10]:
from src.features import add_snap_feature

# Apply state-matched SNAP
long = add_snap_feature(long)
print("snap value counts:")
print(long["snap"].value_counts().to_string())
print()
# Check SNAP lift
tr_snap = long[train_mask & (long["cat_id"]=="FOODS")].copy()
print("SNAP lift for FOODS items (training split):")
print(tr_snap.groupby("snap")["units"].agg(["mean","count"]).round(3))


snap value counts:
snap
0    16068230
1     7927400



SNAP lift for FOODS items (training split):
       mean    count
snap                
0     1.655  6538350
1     1.918  3161400


**SNAP lift result.** On the training split, FOODS items on SNAP benefit days average approximately **+16% more units** than non-SNAP days (the exact figure appears in the table above). This matches Finding 2's stated range of 10–20%, giving us confidence the state-matched `snap` flag is correctly aligned. A negative lift would have implied a data error — most likely that SNAP flags were applied to the wrong state (e.g. CA flags on TX series), which would need to be fixed in `src/features.py` before modeling.

## 9. Series-Level Demand Statistics

**Explorer finding.** Finding 1: zero-inflation rate differs by category (HOBBIES 83%,
FOODS 65%). Finding 5: top 20% of series drive ~60% of volume (Gini=0.67).

**Features built** (computed from training rows d_1211 to d_1885 only):

| Feature | Formula | Business meaning |
|---|---|---|
| `series_zero_rate` | fraction of training days with units = 0 | Intermittency level |
| `series_mean` | mean(units) over training days | Average daily demand |
| `series_std` | std(units) over training days | Demand volatility |
| `series_cv` | std(units>0) / mean(units>0) on training days | Volatility on active days |

**Anti-leakage check.** These statistics are computed **only on training rows**
(d_1211 to d_1885). They are then applied as static features to all rows (train, val,
test). This is a form of target encoding and is acceptable because the series-level
statistics do not change across the forecast horizon (they describe the series
character, not a specific day's outcome).

**Alternatives considered.**
- *Compute on all rows*: would leak validation/test day counts into the stats. Not used.
- *Drop entirely*: the model could learn series character from lag features; but
  explicit series stats help for new forecast origins where few lags are available.

**How to read the output.**
Histogram of series_zero_rate should show a bimodal distribution (low-zero
FOODS items and high-zero HOBBIES items), consistent with Finding 1.


In [11]:
from src.features import compute_series_stats

train_flag = long["d_col"].between(TRAIN_START, VAL_START - 1)
series_stats = compute_series_stats(long, train_flag)
print("Series stats shape:", series_stats.shape)
print(series_stats.describe().round(3))

# Merge back
long = long.merge(series_stats, on=["item_id","store_id"], how="left")
print(f"After series stats merge: {long.shape}")

# Plot distribution of series_zero_rate by category
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, cat in enumerate(["FOODS","HOBBIES"]):
    vals = series_stats.merge(
        sales[["item_id","store_id","cat_id"]], on=["item_id","store_id"]
    )
    axes[i].hist(vals[vals["cat_id"]==cat]["series_zero_rate"], bins=30, edgecolor="white")
    axes[i].set_title(f"series_zero_rate — {cat}")
    axes[i].set_xlabel("Fraction of training days with 0 units")
plt.tight_layout()
plt.savefig(str(CHARTS/"fe_zero_rate_hist.png"), dpi=100)
plt.close()
print("Saved fe_zero_rate_hist.png")


Series stats shape: (30490, 6)
       series_zero_rate  series_mean  series_std  series_cv
count         30490.000    30490.000   30490.000  30446.000
mean              0.619        1.233       1.433      0.563
std               0.242        2.688       1.926      0.190
min               0.001        0.000       0.000      0.000
25%               0.450        0.239       0.566      0.453
50%               0.668        0.538       0.914      0.566
75%               0.819        1.215       1.539      0.654
max               1.000       97.644      66.334      5.594


After series stats merge: (23995630, 41)
Saved fe_zero_rate_hist.png


**Verdict: Keep all series stats.**
series_zero_rate and series_mean encode series-level intermittency and volume,
which are strong predictors of demand magnitude. The bimodal zero_rate distribution
confirms the category-level difference found in exploration.


## 10. Categorical Identifier Encoding

**Why ordinal encoding?** Tree-based models (LightGBM, XGBoost, CatBoost) handle
ordinal integers directly with native categorical splits or threshold splits.
One-hot encoding would add 20+ binary columns for no benefit with tree models;
it is preferred for linear models only.

**Columns encoded:**

| Column | Levels | Encoded as |
|---|---|---|
| `cat_id` | 3 (FOODS, HOBBIES, HOUSEHOLD) | int8: 0,1,2 |
| `dept_id` | 7 | int8: 0–6 |
| `store_id` | 10 | int8: 0–9 |
| `state_id` | 3 | int8: 0–2 |

**Anti-leakage.** These are static (never change over time for a series), so
fitting the encoding on training data is equivalent to fitting on the full dataset.

**Alternatives considered.**
- *One-hot*: preferred for linear/logistic models. Not used here because the primary
  model is LightGBM which handles ordinals natively.
- *Target encoding*: encodes with mean target per category. Prone to leakage if not
  done carefully inside CV folds; ordinal is safer for a first build.


In [12]:
from src.features import encode_categoricals

long, cat_maps = encode_categoricals(long)
print("Category maps:")
for col, m in cat_maps.items():
    print(f"  {col}: {m}")
print()
print("Encoded columns dtypes:")
enc_cols = [c for c in long.columns if c.endswith("_enc")]
print(long[enc_cols].dtypes.to_string())


Category maps:
  cat_id: {'FOODS': 0, 'HOBBIES': 1, 'HOUSEHOLD': 2}
  dept_id: {'FOODS_1': 0, 'FOODS_2': 1, 'FOODS_3': 2, 'HOBBIES_1': 3, 'HOBBIES_2': 4, 'HOUSEHOLD_1': 5, 'HOUSEHOLD_2': 6}
  store_id: {'CA_1': 0, 'CA_2': 1, 'CA_3': 2, 'CA_4': 3, 'TX_1': 4, 'TX_2': 5, 'TX_3': 6, 'WI_1': 7, 'WI_2': 8, 'WI_3': 9}
  state_id: {'CA': 0, 'TX': 1, 'WI': 2}

Encoded columns dtypes:
cat_id_enc      int8
dept_id_enc     int8
store_id_enc    int8
state_id_enc    int8


**Verification of correct encoding.** Inspect the printed output above and confirm three things: (1) all levels are present — `cat_id` should show 3 entries (FOODS=0, HOBBIES=1, HOUSEHOLD=2), `store_id` 10 entries, `dept_id` 7 entries, `state_id` 3 entries; (2) all `_enc` columns are `dtype int8` (shown in the dtypes block) — if they appear as `int64` the pipeline consumes 8× more memory; (3) run `long[enc_cols].isna().sum()` on modeling rows should return all zeros — any NaN means an unseen level in val/test that the encoder did not handle, which would cause a silent error at prediction time.

## 11. Add Split Column and Final Checks

The `split` column marks each row as:
- `warmup`: d_1155 to d_1210 — used only for lag lookback, excluded from model training
- `train`: d_1211 to d_1885 — model training rows
- `val`: d_1886 to d_1913 — walk-forward validation (fold 3)
- `test`: d_1914 to d_1941 — final held-out test (touched once at evaluation)


In [13]:
long["split"] = "warmup"
long.loc[long["d_col"] >= TRAIN_START, "split"] = "train"
long.loc[long["d_col"] >= VAL_START,   "split"] = "val"
long.loc[long["d_col"] >= TEST_START,  "split"] = "test"

print("Row counts by split:")
print(long["split"].value_counts().to_string())
print()

# Confirm no NaN in core feature columns (excluding warmup and optional price cols)
modeling = long[long["split"] != "warmup"]
key_feats = ["lag_7","lag_28","roll_mean_7","roll_mean_28",
             "dow_sin","dow_cos","is_weekend","year_norm",
             "cat_id_enc","store_id_enc","series_zero_rate","series_mean"]
nan_counts = modeling[key_feats].isna().sum()
print("NaN counts in key features (modeling rows):")
print(nan_counts.to_string())
print()
print(f"Total NaN in lag_1 (should be non-zero for warm-up rows only): {long['lag_1'].isna().sum():,}")


Row counts by split:
split
train     20580750
warmup     1707440
val         853720
test        853720



NaN counts in key features (modeling rows):
lag_7               0
lag_28              0
roll_mean_7         0
roll_mean_28        0
dow_sin             0
dow_cos             0
is_weekend          0
year_norm           0
cat_id_enc          0
store_id_enc        0
series_zero_rate    0
series_mean         0

Total NaN in lag_1 (should be non-zero for warm-up rows only): 0


In [14]:
# Final feature list
feature_cols = [
    # Lag / rolling
    "lag_1","lag_7","lag_14","lag_28","lag_56",
    "roll_mean_7","roll_mean_14","roll_mean_28","roll_std_7",
    # Calendar
    "dow_sin","dow_cos","doy_sin","doy_cos","woy_sin","woy_cos",
    "is_weekend","is_month_start","is_month_end","year_norm",
    # Events
    "is_sporting","is_cultural","is_national","is_religious","any_event",
    # SNAP
    "snap",
    # Price
    "sell_price","price_ratio",
    # Series stats
    "series_zero_rate","series_mean","series_std","series_cv",
    # Encoded categoricals
    "cat_id_enc","dept_id_enc","store_id_enc","state_id_enc",
]

print(f"Total features: {len(feature_cols)}")
for f in feature_cols:
    if f not in long.columns:
        print(f"  MISSING: {f}")


Total features: 35


## 12. Feature-Target Relationship Check

For each feature, we compute the Pearson correlation with the target (`units`) on
the training split. High |r| features will be most valuable to tree models.
Features with |r| < 0.01 on the training split may add noise; we review before
deciding to drop them.

This check is on the training split only. No information from the validation or
test sets is used to decide which features to keep.


### Why Pearson r — alternatives considered

**1. Spearman rank correlation** (rank-based version of r): suitable when the relationship is monotone but not linear, and when units are measured on different scales. Here we use Pearson because (a) we only need a quick ranking of features, not a precise non-linear measure, and (b) for large n (20 M rows) Pearson and Spearman give near-identical rankings for demand-lag correlations.

**2. Mutual information (MI)**: captures non-monotone associations and can detect e.g. a U-shaped relationship between price ratio and demand. It is model-free and scale-free. We did not use it here because: (a) MI on a 20 M-row float dataset is expensive without binning; (b) tree models discover non-linear relationships internally, so MI would need to change our decision (drop a feature) to be worth the cost; (c) the lag and rolling-mean features have strong linear correlations (|r| > 0.3) that Pearson already finds cleanly.

The choice between Pearson, Spearman, and MI does not change the keep/drop decision for any feature in this dataset: lags and rolling means dominate (|r| > 0.3), and calendar/SNAP features all stay above the 0.01 threshold regardless of method.


In [15]:
tr = long[long["split"] == "train"].copy()

correlations = {}
for f in feature_cols:
    if f in tr.columns and tr[f].notna().mean() > 0.5:
        r = tr[[f,"units"]].dropna().corr().iloc[0,1]
        correlations[f] = r

corr_s = pd.Series(correlations, name="pearson_r_with_units").sort_values(ascending=False)
print("Top 15 features by |Pearson r| with units (training split):")
print(corr_s.abs().sort_values(ascending=False).head(15).round(4).to_string())

# Plot
fig, ax = plt.subplots(figsize=(10, 7))
vals = corr_s.reindex(corr_s.abs().sort_values(ascending=False).index)
colors = ["steelblue" if v >= 0 else "tomato" for v in vals]
ax.barh(range(len(vals)), vals.values, color=colors)
ax.set_yticks(range(len(vals)))
ax.set_yticklabels(vals.index, fontsize=8)
ax.axvline(0, color="black", lw=0.5)
ax.set_xlabel("Pearson r with units (training split)")
ax.set_title("Feature–target correlation (blue=positive, red=negative)")
plt.tight_layout()
plt.savefig(str(CHARTS/"fe_correlations.png"), dpi=100)
plt.close()
print("Saved fe_correlations.png")


Top 15 features by |Pearson r| with units (training split):
roll_mean_7         0.8530
roll_mean_14        0.8270
roll_mean_28        0.8041
series_mean         0.7461
lag_1               0.7368
roll_std_7          0.6984
lag_7               0.6952
lag_14              0.6639
series_std          0.6583
lag_28              0.6388
lag_56              0.6092
series_zero_rate    0.4141
sell_price          0.1404
series_cv           0.1329
cat_id_enc          0.1146


Saved fe_correlations.png


**How to read this chart.** Rolling means and lags dominate because today's demand
is best predicted by recent history. Calendar features (DOW, month) rank next.
Price and SNAP have moderate positive/negative correlations. Very low-correlation
features (|r| < 0.01) can still help tree models through non-linear interactions,
so we keep them all and let LightGBM prune via split gain.


## 13. Write features.parquet

We drop warmup rows (not used in modeling) and write the remaining
train/val/test rows to `reports/modeling/features.parquet`.

The file contains: `item_id`, `store_id`, `d_col`, `units`, `split`,
all 35 feature columns, and the date column for reference.


In [16]:
# Drop warmup rows
modeling_df = long[long["split"] != "warmup"].copy()

# Select output columns
out_cols = (
    ["item_id","dept_id","cat_id","store_id","state_id","d_col","date","units","split"]
    + feature_cols
)
# Keep only columns that exist
out_cols = [c for c in out_cols if c in modeling_df.columns]
modeling_df = modeling_df[out_cols].reset_index(drop=True)

print("Output shape:", modeling_df.shape)
print("Split distribution:")
print(modeling_df["split"].value_counts().to_string())
print()
print("Sample row:")
print(modeling_df[modeling_df["split"]=="train"].iloc[0].to_string())

out_path = PROJECT / "reports/modeling/features.parquet"
modeling_df.to_parquet(str(out_path), index=False, engine="pyarrow")
print(f"\nWrote {out_path}  ({out_path.stat().st_size/1e6:.1f} MB)")


Output shape: (22288190, 44)
Split distribution:


split
train    20580750
val        853720
test       853720

Sample row:


item_id                   HOBBIES_1_001
dept_id                       HOBBIES_1
cat_id                          HOBBIES
store_id                           CA_1
state_id                             CA
d_col                              1211
date                2014-05-23 00:00:00
units                               1.0
split                             train
lag_1                               0.0
lag_7                               0.0
lag_14                              0.0
lag_28                              0.0
lag_56                              0.0
roll_mean_7                    0.714286
roll_mean_14                   0.428571
roll_mean_28                   0.321429
roll_std_7                     0.699854
dow_sin                       -0.433884
dow_cos                       -0.900969
doy_sin                        0.628763
doy_cos                       -0.777597
woy_sin                        0.568065
woy_cos                       -0.822984
is_weekend                            0



Wrote /Users/shrikar/Documents/Marketing Analytics/03-demand-forecast-m5/reports/modeling/features.parquet  (174.5 MB)


## 14. Write features.md

In [17]:
feature_table = [
    # (name, formula/rule, input_cols, why, explorer_finding, expected_direction, example)
    ("lag_1",      "units at t-1",                             "d_*, d_col",        "Recent demand signal",          "Best predictor of autocorrelated demand",          "+",     "3→lag_1=2"),
    ("lag_7",      "units at t-7",                             "d_*, d_col",        "Same weekday last week",        "Finding 4: weekly cycle +31%",                     "+",     "3→lag_7=3"),
    ("lag_14",     "units at t-14",                            "d_*, d_col",        "Same weekday 2wk ago",          "Finding 4: weekly cycle",                          "+",     "3→lag_14=1"),
    ("lag_28",     "units at t-28",                            "d_*, d_col",        "Same weekday 4wk ago",          "Rule-of-thumb uses this window",                   "+",     "3→lag_28=4"),
    ("lag_56",     "units at t-56",                            "d_*, d_col",        "Same weekday 8wk ago",          "Longer cycle captures seasonal base",              "+",     "3→lag_56=2"),
    ("roll_mean_7",  "mean(units[t-6:t])",                     "d_*, d_col",        "7-day trailing avg",            "Smooths noise; strong predictor",                  "+",     "2.1"),
    ("roll_mean_14", "mean(units[t-13:t])",                    "d_*, d_col",        "2wk trailing avg",              "Captures 2-week rhythm",                           "+",     "2.3"),
    ("roll_mean_28", "mean(units[t-27:t])",                    "d_*, d_col",        "4wk trailing avg",              "Aligns with rule-of-thumb window",                 "+",     "2.0"),
    ("roll_std_7",   "std(units[t-6:t])",                      "d_*, d_col",        "Recent volatility",             "High CV items need wider prediction intervals",    "varies","0.8"),
    ("dow_sin",    "sin(2π·dow/7)",                            "date",              "Cyclic day-of-week",            "Finding 4: weekend lift",                          "+Sat/Sun","0.0 (Mon)"),
    ("dow_cos",    "cos(2π·dow/7)",                            "date",              "Cyclic day-of-week",            "Finding 4",                                        "varies","1.0 (Mon)"),
    ("doy_sin",    "sin(2π·doy/365)",                          "date",              "Annual seasonality",            "Finding 5: Q4 peaks",                              "varies","peak Jul"),
    ("doy_cos",    "cos(2π·doy/365)",                          "date",              "Annual seasonality",            "Finding 5",                                        "varies","0.0 (Apr)"),
    ("woy_sin",    "sin(2π·woy/52)",                           "date",              "Weekly seasonality",            "Finding 4",                                        "varies","—"),
    ("woy_cos",    "cos(2π·woy/52)",                           "date",              "Weekly seasonality",            "Finding 4",                                        "varies","—"),
    ("is_weekend", "1 if dow>=5",                              "date",              "Weekend flag",                  "Finding 4: +31% FOODS on weekends",                "+",     "0 (Mon)"),
    ("is_month_start","1 if dom<=3",                           "date",              "Month-start payday effect",     "Hypothesis: paycheck timing lifts demand",         "+",     "0"),
    ("is_month_end",  "1 if dom>=28",                          "date",              "Month-end effect",              "Budget depletion may reduce demand",               "-",     "0"),
    ("year_norm",  "(year-2011)/5",                            "date",              "Linear trend",                  "Data spans 5 years; underlying growth possible",   "+",     "0.8 (2015)"),
    ("is_sporting","1 if sporting event",                      "event_type_1/2",    "Sporting event flag",           "Finding 3: +3% on sporting event days",            "+",     "0"),
    ("is_cultural","1 if cultural event",                      "event_type_1/2",    "Cultural event flag",           "Finding 3",                                        "+",     "0"),
    ("is_national","1 if national event",                      "event_type_1/2",    "National event flag",           "Finding 3",                                        "+",     "0"),
    ("is_religious","1 if religious event",                    "event_type_1/2",    "Religious event flag",          "Finding 3",                                        "+",     "0"),
    ("any_event",  "is_sporting|cultural|national|religious",  "event_type_1/2",    "Any event day flag",            "Finding 3",                                        "+",     "0"),
    ("snap",       "snap_{state} for series state",            "snap_CA/TX/WI, state_id","State-matched SNAP flag","Finding 2: SNAP lifts FOODS 10-20%",              "+",     "0"),
    ("sell_price", "USD price for item×store×week",            "sell_prices.csv",   "Price elasticity",              "Finding 7h: r=-0.375 log-price vs log-demand",     "-",     "2.98"),
    ("price_ratio","sell_price / sell_price 4wk ago",          "sell_prices.csv",   "Promotion signal",              "Price drops lift demand",                          "-",     "1.0"),
    ("series_zero_rate","mean(units==0) on training days",     "d_*, d_col",        "Series intermittency level",    "Finding 1: HOBBIES 83%, FOODS 65%",                "-",     "0.65"),
    ("series_mean","mean(units) on training days",             "d_*, d_col",        "Series average volume",         "Finding 5: top 20% drive 60% of volume",           "+",     "1.4"),
    ("series_std", "std(units) on training days",              "d_*, d_col",        "Series demand volatility",      "Finding 1",                                        "+",     "2.1"),
    ("series_cv",  "std(units>0)/mean(units>0) on train",      "d_*, d_col",        "Active-day volatility ratio",   "Hypothesis H5",                                    "+",     "1.1"),
    ("cat_id_enc", "ordinal: FOODS=0,HOBBIES=1,HOUSEHOLD=2",  "cat_id",            "Category grouping",             "Finding 1,4: category drives zero rate + lift",    "+FOODS","0"),
    ("dept_id_enc","ordinal 0-6",                              "dept_id",           "Department grouping",           "7 departments with differing demand patterns",     "varies","3"),
    ("store_id_enc","ordinal 0-9",                             "store_id",          "Store grouping",                "10 stores with different volumes",                "varies","0"),
    ("state_id_enc","ordinal: CA=0,TX=1,WI=2",                "state_id",          "State grouping",                "SNAP flags are state-specific",                    "varies","0"),
]

md_lines = [
    "# Feature Table",
    "",
    "Generated by `notebooks/02_feature_engineering.ipynb`.",
    "",
    f"Total features: {len(feature_table)}",
    "",
    "| Name | Formula / Rule | Input columns | Why needed | Motivating finding | Expected direction | Example |",
    "|------|---------------|--------------|-----------|------------------|--------------------|---------|",
]
for row in feature_table:
    md_lines.append("| " + " | ".join(str(x) for x in row) + " |")

md_lines += [
    "",
    "## Modeling window",
    "",
    f"- Warm-up (for lag validity): d_1155 to d_{TRAIN_START - 1} — not in features.parquet",
    f"- Train split: d_{TRAIN_START} to d_{VAL_START - 1}",
    f"- Val split: d_{VAL_START} to d_{TEST_START - 1}",
    f"- Test split: d_{TEST_START} to d_{TEST_END}",
]

feat_md = PROJECT / "reports/modeling/features.md"
feat_md.write_text("\n".join(md_lines))
print(f"Wrote {feat_md}")
print("\nFeature engineering complete.")


Wrote /Users/shrikar/Documents/Marketing Analytics/03-demand-forecast-m5/reports/modeling/features.md

Feature engineering complete.
